# Feature Importance Analysis via Progressive Feature Removal
## Refactored Object-Oriented Implementation (Unified Trainer API)

This notebook implements an iterative feature pruning sweep to analyze feature importance using Lexos' unified classification interface.

**Strategy:**
1. Initialize the `Classifier` with `features="all"` to automatically discover baseline corpus statistics features.
2. Extract the dynamically discovered feature list directly from the classifier instance.
3. Iteratively remove one feature at a time in a randomized sequence.
4. Track performance degradation or improvement metrics across each configuration by setting explicit feature subsets.

In [1]:
"""Cell 1: Library Imports and Unified Architecture Hooks."""
from pathlib import Path
import random
import numpy as np
import pandas as pd

# Import the new object-oriented Lexos components
from lexos.classification import Classifier, MLPPipeline
from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer

In [2]:
"""Cell 2: Seed Initialization and Data Directory Resolution."""
SEED = 42
rng = random.Random(SEED)
np.random.seed(SEED)

# Setup cleaning pipelines
scrubber = Scrubber()
scrubber.add_pipe("lower_case")
scrubber.add_pipe("digits")
scrubber.add_pipe("punctuation")

tokenizer = Tokenizer(model="en_core_web_sm")

# Resolve absolute pathing structures dynamically
base = Path.cwd()
search_roots = [base] + list(base.parents)
data_dir = next(
    (root / "fed_papers" for root in search_roots if (root / "fed_papers").exists()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Could not locate 'fed_papers' from the current root pathing chain.")

train_dirs = ["HAMILTON", "MADISON"]
unknown_dirs = ["DISPUTED", "COAUTHORED"]

train_files = []
train_labels = []
for author in train_dirs:
    files = sorted((data_dir / author).glob("*.txt"))
    train_files.extend(files)
    train_labels.extend([author] * len(files))

unknown_files = []
for subset in unknown_dirs:
    unknown_files.extend(sorted((data_dir / subset).glob("*.txt")))

train_texts = [path.read_text(encoding="utf-8", errors="ignore") for path in train_files]
unknown_texts = [path.read_text(encoding="utf-8", errors="ignore") for path in unknown_files]
train_ids = [path.name for path in train_files]
unknown_ids = [path.name for path in unknown_files]

print(f"Using data directory: {data_dir}")
print(f"Training docs: {len(train_texts)}")
print(f"Unknown docs: {len(unknown_texts)}")
print(pd.Series(train_labels).value_counts())

Using data directory: /home/mango/Lexos_Independant_Research/lexos/doc_src/docs/tutorials/classification/fed_papers
Training docs: 65
Unknown docs: 15
HAMILTON    51
MADISON     14
Name: count, dtype: int64


In [3]:
"""Cell 3: Baseline Initialization and Dynamic Feature Discovery."""
# Define the baseline training configuration strategy profile
baseline_strategy = MLPPipeline(
    seed=SEED,
    min_df=2,
    test_size=0.2,
    cv_splits=5,
    include_bigrams=True,
    use_smote=True,
    mlp_kwargs={
        "hidden_layer_sizes": (64,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 1000,
    },
)

# Instantiate a temporary baseline classifier to dynamically extract available corpus stats features
baseline_discoverer = Classifier(
    train_data=train_texts,
    labels=train_labels,
    pipeline=baseline_strategy,
    features="all" # Automatically triggers build_corpus_stat_features internally
)

# Call the build method explicitly to get the full list for our experiment sweep
all_corpus_stat_features = baseline_discoverer.build_corpus_stat_features()
feature_removal_order = all_corpus_stat_features.copy()
rng.shuffle(feature_removal_order) # Still need to implement different algorithms

print(f"CorpusStats features dynamically discovered: {len(all_corpus_stat_features)}")
print("Random removal order:\n", ", ".join(feature_removal_order))

CorpusStats features dynamically discovered: 40
Random removal order:
 yule_k, total_terms, hapax_legomena, propn_count, noun_count, punc_count, aux_count, hapax_dislegomena, average_word_length, det_count, average_sentence_length, adj_count, sentence_count, emotion_word_count, hapax_legomenon_rate, part_count, pron_count, verb_count, question_count, flesch_reading_ease, adverb_count, cconj_count, sconj_count, adp_count, stop_word_count, polarity, participle_count, intj_count, character_count, total_tokens, sym_count, num_count, ttr, nominal_ratio, guiraud_index, exclamation_count, vocabulary_density, subjectivity, unique_word_count, simple_nominal_ratio


In [ ]:
# Setup your strategy profile and request a randomized feature removal sweep
mlp_strategy = MLPPipeline(
    seed=SEED,
    feature_removal="random",  # Natively triggers randomized pruning loops
    mlp_kwargs={"hidden_layer_sizes": (64,), "max_iter": 1000}
)

# Initialize the classifier with 'all' features
classifier = Classifier(
    train_data=train_texts,
    labels=train_labels,
    pipeline=mlp_strategy,
    features="all"
)

# Run the entire sweep
sweep_results_df = classifier.feature_importance_sweep()

# Display the final summary report directly
print(sweep_results_df)

In [ ]:
"""Cell 5: Delta Profiling and Summary Analysis Extraction."""
results_df = pd.DataFrame(experiment_rows)
baseline_metrics = results_df.loc[results_df["configuration"] == "baseline"].iloc[0]

metric_cols = [
    "holdout_accuracy",
    "holdout_balanced_accuracy",
    "holdout_macro_f1",
    "cv_accuracy",
    "cv_balanced_accuracy",
    "cv_macro_f1",
]

# Calculate deviation matrices vs baseline configuration thresholds
for metric_name in metric_cols:
    results_df[f"{metric_name}_delta_vs_baseline"] = results_df[metric_name] - float(baseline_metrics[metric_name])

summary_columns = [
    "configuration",
    "removed_feature",
    "features_remaining",
    "holdout_accuracy",
    "holdout_balanced_accuracy",
    "holdout_macro_f1",
    "cv_accuracy",
    "holdout_macro_f1_delta_vs_baseline",
]

print("\n--- Feature Removal Pruning Summary Report ---")
print(results_df[summary_columns].to_string(index=False))

# Extract the most critical degradation tipping points
impact_row = results_df.loc[results_df["holdout_macro_f1_delta_vs_baseline"].idxmin()]
print(f"\nGreatest holdout macro-F1 drop vs baseline:\n{impact_row['removed_feature']} | delta={impact_row['holdout_macro_f1_delta_vs_baseline']:.4f}")

best_row = results_df.loc[results_df["holdout_macro_f1"].idxmax()]
print(f"\nBest performing feature layout combination:\n{best_row['configuration']} | removed={best_row['removed_feature']} | features_remaining={int(best_row['features_remaining'])} | F1={best_row['holdout_macro_f1']:.4f}")